#compared to latest gpt models this will be basic

In [1]:
#Self attention-generates the output with respect to the context of all the input tokens....selctive access
#Tokens are assigned importance based on weights(attention_scores)
#attention scores=unnormalized attention wts()

import torch

inputs=torch.tensor(
    [
        [0.43,0.15,0.89],#Your
        [0.55,0.87,0.66],#journey
        [0.57,0.85,0.64],#starts
        [0.22,0.58,0.33],#with
        [0.77,0.25,0.10],#one
        [0.05,0.80,0.55] #step
    ]
)

In [2]:
input_query=inputs[1]
input_query


tensor([0.5500, 0.8700, 0.6600])

In [3]:
#What happens
0.55*0.43+0.87*0.15+0.66*0.89

0.9544

In [10]:
res=0.
for j,ele in enumerate(inputs[1]):
    res+=inputs[0][j]*input_query[j]
res

tensor(0.9544)

In [15]:
r=[]
for i in range(0,6):
    res=0.
    for j,ele in enumerate(inputs[i]):
        res+=inputs[i][j]*input_query[j]
        r.append(res)
r

[tensor(0.9544),
 tensor(0.9544),
 tensor(0.9544),
 tensor(1.4950),
 tensor(1.4950),
 tensor(1.4950),
 tensor(1.4754),
 tensor(1.4754),
 tensor(1.4754),
 tensor(0.8434),
 tensor(0.8434),
 tensor(0.8434),
 tensor(0.7070),
 tensor(0.7070),
 tensor(0.7070),
 tensor(1.0865),
 tensor(1.0865),
 tensor(1.0865)]

In [17]:
i=3
for j,ele in enumerate(inputs[i]):
    res=torch.dot(input_query,inputs[0])
res

tensor(0.9544)

In [ ]:
query=inputs[1]
attn_scores_2=torch.empty(inputs.shape[0]) #tensor([0.,0.,0.,0.])
for i,x_i in enumerate(inputs):
    attn_scores_2[i]=torch.dot(x_i,query)     #score[i]=x[i]*query
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [19]:
attn_wt2_temp=attn_scores_2/attn_scores_2.sum()
attn_wt2_temp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [21]:
attn_wt2_temp.sum()

tensor(1.0000)

In [22]:
def softmax_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)
softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [23]:
torch.softmax(attn_scores_2,dim=0)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [24]:
query=inputs[1]

context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+=attn_wt2_temp[i]*x_i
print(context_vec_2)

tensor([0.4355, 0.6451, 0.5680])


In [26]:
for i,x_i in enumerate(inputs):
    print("Index:",i,x_i)

Index: 0 tensor([0.4300, 0.1500, 0.8900])
Index: 1 tensor([0.5500, 0.8700, 0.6600])
Index: 2 tensor([0.5700, 0.8500, 0.6400])
Index: 3 tensor([0.2200, 0.5800, 0.3300])
Index: 4 tensor([0.7700, 0.2500, 0.1000])
Index: 5 tensor([0.0500, 0.8000, 0.5500])


Self attention without Trainable Weights

In [ ]:
# query=inputs[1]
attn_scores=torch.empty(6,6)
for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j]=torch.dot(x_i,x_j)
print(attn_scores) #Not normalized yet

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [28]:
#instead of for loops we can do matrix multiplication that can be efficient
attn_scores=inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [29]:
attn_weights=torch.softmax(attn_scores,dim=1)#sum of rows=1
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [30]:
all_context_vecs=attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

Computing attention weights step by step with trainable weights

In [31]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [32]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2


In [33]:
torch.manual_seed(123)

W_query=torch.nn.Parameter(torch.rand(d_in,d_out))#d-dimension
W_key=torch.nn.Parameter(torch.rand(d_in,d_out))
W_value=torch.nn.Parameter(torch.rand(d_in,d_out))
query_2=x_2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [35]:
keys=inputs @ W_key
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [37]:
value=inputs @W_value
value
value.shape

torch.Size([6, 2])